# AURA Stage 8.1 — AppWorld Recovery & Native Agent Pilot (v6)

**Purpose:** Continue the *missing external benchmark part* of Stage 8.1 without rerunning AURA's already-audited local tests. This notebook runs the **official AppWorld** setup and a genuine model-directed agent on three **dev** tasks, with the official AppWorld evaluator.

**Important scientific boundary:** AppWorld's native evaluator is *not* an AURA independent verification certificate. The AppWorld-to-AURA contract/evidence adapter remains future work. No publication-level score is claimed.

**Why this version:** The prior Kaggle session reset (`ENVIRONMENT_NOT_FOUND`) and the previous AppWorld import/development-index preflight failed. Here setup happens **before loading any GPU model**, in an isolated managed **Python 3.11** interpreter. Every failed setup check stores its full Python traceback in the small final ZIP; the notebook does not merely return `exit code 1`.

**Run All from a NEW Kaggle notebook with Internet and GPU enabled.** The final cell creates `/kaggle/working/AURA_APPWORLD_STAGE8_1_v6_RESULTS.zip` even when AppWorld setup fails (provided Kaggle can execute the final cell). Upload that single ZIP for analysis. Model inference, not training/fine-tuning.

**Official method:** [AppWorld GitHub installation and benchmark documentation](https://github.com/StonyBrookNLP/appworld/blob/main/README.md).

## 1 — Frozen settings and private staging area
The output archive contains only curated research records and, if needed, a bounded traceback. Downloaded packages, temporary AppWorld databases, model weights, and hidden evaluator details are excluded.

In [ ]:
import os, sys, re, json, time, hashlib, zipfile, tempfile, subprocess, traceback, csv, random, shutil, importlib.metadata, platform
from pathlib import Path

SEED=20261025
MODEL_ID='Qwen/Qwen2.5-3B-Instruct'
SPLIT='dev'  # development set, NOT held-out benchmark evaluation
N_TASKS=3
MAX_STEPS=7
MAX_MODEL_NEW_TOKENS=300
EXPERIMENT='aura_stage8_1_appworld_v6_dev'
RUN_MODEL=True  # only used AFTER all native AppWorld setup tests pass

BASE=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
TMPROOT=Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path(tempfile.gettempdir())
WORK=Path(tempfile.mkdtemp(prefix='aura_appworld_v6_',dir=TMPROOT))
OUT=WORK/'private_results'; OUT.mkdir(parents=True,exist_ok=True)
ZIP_PATH=BASE/'AURA_APPWORLD_STAGE8_1_v6_RESULTS.zip'
APPWORLD_ROOT=WORK/'appworld_root';APPWORLD_ROOT.mkdir()
APPWORLD_VENV=WORK/'appworld_py311'
APPWORLD_PY=APPWORLD_VENV/'bin'/'python'
APPWORLD_READY=False
MODEL_READY=False
PROBE_TASK_ID=None
random.seed(SEED)
CONFIG={'seed':SEED,'model_id':MODEL_ID,'split':SPLIT,'task_selection':'first_three_published_dev_order',
        'n_tasks':N_TASKS,'max_steps':MAX_STEPS,'max_model_new_tokens':MAX_MODEL_NEW_TOKENS,
        'run_model':RUN_MODEL,'experiment_name':EXPERIMENT,'track_a_local_results_repeated':False,
        'external_aura_certificates':False,'held_out_score_claimed':False,'appworld_python_requested':'3.11',
        'notebook_version':'v6-appworld-only', 'main_python':platform.python_version()}
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2))
STEPS=[]; FAILURE=None; APPWORLD_VERSION=None
print('NEW recovery environment:', WORK)
print('Only deliverable:', ZIP_PATH)

## 2 — Isolated Python 3.11 AppWorld installation, with explicit error capture
This stage does not import AppWorld into Kaggle's main kernel. A failed setup stops further expensive inference, but **does not stop the notebook from packaging a readable diagnostic**.

In [ ]:
ENV=os.environ.copy()
for _key in ('PYTHONPATH','PYTHONHOME','VIRTUAL_ENV'):
    ENV.pop(_key,None)
ENV['APPWORLD_ROOT']=str(APPWORLD_ROOT)
ENV['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'

def run_checked(label,cmd,timeout=1800,env=ENV):
    global FAILURE
    print('CHECK:',label)
    t=time.perf_counter()
    try:
        p=subprocess.run(list(map(str,cmd)),text=True,stdout=subprocess.PIPE,
                         stderr=subprocess.PIPE,timeout=timeout,env=env)
        elapsed=round(time.perf_counter()-t,2)
        STEPS.append({'step':label,'exit_code':p.returncode,'elapsed_s':elapsed,'status':'PASS' if p.returncode==0 else 'FAIL'})
        if p.returncode:
            detail=(f'FAILED STEP: {label}\nCOMMAND: {list(map(str,cmd))}\nRETURN CODE: {p.returncode}\n'
                    f'STDOUT:\n{p.stdout[-16000:]}\nSTDERR:\n{p.stderr[-26000:]}\n')
            FAILURE={'phase':label,'kind':'nonzero_exit','return_code':p.returncode,'details':detail[-35000:]}
            print(detail[-5000:])
            return None
        if p.stdout.strip():print(p.stdout[-800:])
        return p
    except Exception as e:
        STEPS.append({'step':label,'exit_code':None,'status':'EXCEPTION','exception_type':type(e).__name__})
        FAILURE={'phase':label,'kind':'exception','return_code':None,'details':traceback.format_exc()[-35000:]}
        print('Check exception',label,repr(e))
        return None

def is_ok(label,cmd,timeout=1800,env=ENV):
    return run_checked(label,cmd,timeout=timeout,env=env) is not None

# Only a handful of explicit, ordered setup phases. No model is downloaded here.
try:
    uv_cmd=[sys.executable,'-m','uv']
    if not is_ok('uv_available',uv_cmd+['--version'],timeout=40):
        # `uv` has no runtime dependencies and does not replace Pydantic/FastAPI.
        if is_ok('install_uv',[sys.executable,'-m','pip','install','--no-deps','uv>=0.9,<1'],timeout=600,env=None):
            FAILURE=None
            is_ok('verify_uv',uv_cmd+['--version'],timeout=50)
    if FAILURE is None:
        is_ok('install_managed_python_311',uv_cmd+['python','install','3.11'],timeout=1100)
    if FAILURE is None:
        is_ok('create_python311_venv',uv_cmd+['venv','--python','3.11',str(APPWORLD_VENV)],timeout=170)
    if FAILURE is None and not APPWORLD_PY.is_file():
        FAILURE={'phase':'python_venv_exists','kind':'file_missing','details':str(APPWORLD_PY)}
    if FAILURE is None:
        is_ok('verify_python311',[APPWORLD_PY,'-c','import sys; print(sys.version); assert sys.version_info[:2] == (3,11)'],timeout=45)
    if FAILURE is None:
        is_ok('install_official_appworld',uv_cmd+['pip','install','--python',str(APPWORLD_PY),'appworld'],timeout=2100)
    if FAILURE is None:
        is_ok('appworld_unpack_official_code',[APPWORLD_PY,'-m','appworld.cli','install'],timeout=1300)
    if FAILURE is None:
        is_ok('appworld_download_official_data',[APPWORLD_PY,'-m','appworld.cli','download','data'],timeout=3000)
    if FAILURE is None:
        is_ok('appworld_python_import',[APPWORLD_PY,'-c','import appworld; from appworld import AppWorld; print("APPWORLD_IMPORT_OK")'],timeout=130)
    if FAILURE is None:
        is_ok('appworld_dev_task_index',[APPWORLD_PY,'-c',
             'from appworld import load_task_ids; ids=list(load_task_ids("dev")); print("DEV_TASK_COUNT",len(ids)); assert len(ids)>=3'],timeout=180)
    if FAILURE is None:
        r=run_checked('record_appworld_version',[APPWORLD_PY,'-c',
               'import importlib.metadata as m;print(m.version("appworld"))'],timeout=60)
        if r: APPWORLD_VERSION=r.stdout.strip()
    # The preceding failure from Kaggle v5 is now separately diagnosed.
    APPWORLD_READY=FAILURE is None
except Exception:
    FAILURE={'phase':'setup_unsafe_exception','kind':'exception','details':traceback.format_exc()[-35000:]}
    APPWORLD_READY=False

(OUT/'setup_checklist.json').write_text(json.dumps(STEPS,indent=2))
(OUT/'setup_status.json').write_text(json.dumps({
    'ready_to_initialize_task':APPWORLD_READY,'failure_phase':FAILURE['phase'] if FAILURE else None,
    'appworld_version':APPWORLD_VERSION,'installed_python':str(APPWORLD_PY) if APPWORLD_READY else None},indent=2))
if FAILURE:
    (OUT/'SETUP_DIAGNOSTIC.txt').write_text(FAILURE['details'],encoding='utf-8')
    print('APPWORLD_SETUP_FAILED:',FAILURE['phase'])
else:
    print('AppWorld installation + package import + dev index PASS on Python 3.11')

## 3 — Initialize one real AppWorld world and test the **native evaluator**
This is a **no-agent control**, not a model success. It verifies that loading tasks and scoring truly work before any model download. Error details are saved if it fails.

In [ ]:
PRECHECK={'executed':False,'world_load_ok':False,'native_evaluator_ok':False,'control_success':None}
if APPWORLD_READY:
    # Execute through the isolated interpreter; capture original traceback, not just exit=1.
    code='import json, traceback\nfrom appworld import AppWorld, load_task_ids\nids=list(load_task_ids("dev")); tid=ids[0]\nwith AppWorld(task_id=tid, experiment_name="aura_stage8_1_preflight_control") as world:\n    print("INIT_OK", tid)\n    print("INSTRUCTION_SIZE",len(str(world.task.instruction)))\n    score=world.evaluate().to_dict()\n    print("NATIVE_NO_OP_SCORE",json.dumps({"success":score.get("success"),"num_tests":score.get("num_tests")}))\n'
    r=run_checked('first_dev_world_and_native_noop_evaluator',[APPWORLD_PY,'-c',code],timeout=300)
    PRECHECK['executed']=True
    if r:
        PRECHECK.update(world_load_ok=True,native_evaluator_ok=True,
                        control_stdout=r.stdout[-1800:])
    else:
        APPWORLD_READY=False
        (OUT/'SETUP_DIAGNOSTIC.txt').write_text(FAILURE['details'],encoding='utf-8')
(OUT/'native_evaluator_preflight.json').write_text(json.dumps(PRECHECK,indent=2))
(OUT/'setup_checklist.json').write_text(json.dumps(STEPS,indent=2))
(OUT/'setup_status.json').write_text(json.dumps({'ready_for_native_agent':APPWORLD_READY,
  'failure_phase':FAILURE.get('phase') if FAILURE else None,
  'appworld_version':APPWORLD_VERSION,'first_world_and_native_evaluator_passed':PRECHECK['native_evaluator_ok']},indent=2))
print('Real AppWorld world + evaluator preflight:',PRECHECK['native_evaluator_ok'])

## 4 — Load Qwen only if AppWorld passed **all** setup checks
The model chooses code actions; answers are **not scripted**. If the model is unavailable, the notebook returns an honest partial-results ZIP.

In [ ]:
MODEL_ERROR=None
MODEL_READY=False
if APPWORLD_READY and RUN_MODEL:
    try:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA unavailable; real model inference skipped rather than claiming CPU benchmark results')
        print('GPU',torch.cuda.get_device_name(0))
        # No heavy model dependencies are installed until native benchmark is operational.
        try:
            import transformers,accelerate
        except (ImportError,ModuleNotFoundError):
            install=run_checked('install_controller_transformers',[sys.executable,'-m','pip','install','-q',
               'transformers>=4.46,<5','accelerate>=0.34','sentencepiece','safetensors'],timeout=1000,env=None)
            if install is None: raise RuntimeError('Could not install model inference dependencies')
        from transformers import AutoTokenizer,AutoModelForCausalLM
        import torch
        torch.manual_seed(SEED)
        tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=torch.float16,device_map='auto',trust_remote_code=False)
        model.eval()
        MODEL_READY=True
    except Exception as exc:
        MODEL_ERROR=repr(exc)
        (OUT/'MODEL_DIAGNOSTIC.txt').write_text(traceback.format_exc()[-25000:],encoding='utf-8')
        print('MODEL_UNAVAILABLE:',MODEL_ERROR)
else:
    MODEL_ERROR='AppWorld not ready or run_model=false'

def llm_chat(messages,max_new_tokens=MAX_MODEL_NEW_TOKENS):
    if not MODEL_READY:raise RuntimeError('model_unavailable')
    keep=messages if len(messages)<=8 else [messages[0],messages[1]]+messages[-6:]
    formatted=tok.apply_chat_template(keep,tokenize=False,add_generation_prompt=True)
    inp=tok(formatted,return_tensors='pt',truncation=True,max_length=7500)
    dev=next(model.parameters()).device
    inp={k:v.to(dev) for k,v in inp.items()}
    t0=time.perf_counter()
    with torch.inference_mode():
        generated=model.generate(**inp,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id)
    reply=tok.decode(generated[0][inp['input_ids'].shape[-1]:],skip_special_tokens=True)
    return reply,round(time.perf_counter()-t0,4)

(OUT/'model_status.json').write_text(json.dumps({'model_id':MODEL_ID,'ready':MODEL_READY,
    'error':MODEL_ERROR,'inference_only':True,'training':False},indent=2))
print('Model ready:',MODEL_READY)

## 5 — Execute genuine AppWorld agent tasks and native no-action controls
A dedicated AppWorld worker, running under Python 3.11, receives only the agent's prompts/responses. Native evaluation occurs **after** the agent finishes and is never passed back into its prompts. The worker records trajectories, official success counts, and selected API evidence.

In [ ]:
WORKER_SOURCE='"""AppWorld side of AURA Stage 8.1. Runs only in an isolated Python venv.\n\nNo language model is installed here: language-model responses arrive over a\nline-delimited JSON protocol from the parent Kaggle notebook. Task ground truth\nand native evaluator feedback never enter that request channel.\n"""\nimport sys, os, json, re, ast, traceback, hashlib, shutil, csv, contextlib\nfrom pathlib import Path\n\nOUT=Path(os.environ[\'AURA_RESULTS_DIR\'])\nROOT=Path(os.environ[\'APPWORLD_ROOT\'])\nSPLIT=os.environ.get(\'AURA_APPWORLD_SPLIT\',\'dev\')\nCOUNT=int(os.environ.get(\'AURA_APPWORLD_COUNT\',\'3\'))\nSTEPS=int(os.environ.get(\'AURA_APPWORLD_STEPS\',\'7\'))\nEXPERIMENT=os.environ.get(\'AURA_APPWORLD_EXPERIMENT\',\'aura_stage8_1_external\')\nDEST=OUT/\'track_B_native_AppWorld\'; DEST.mkdir(parents=True,exist_ok=True)\nSYSTEM=(\'You are a coding agent working in AppWorld. Solve the user task using only \'\n        \'the documented AppWorld APIs. Return one fenced Python block each turn, \'\n        \'containing apis.<app>.<api>(...) calls. Respect the environment safety rules. \'\n        \'Do not access hidden evaluation data or host filesystem. Call \'\n        \'apis.supervisor.complete_task() only after doing the requested work.\')\n\ndef write_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding=\'utf-8\')\n\ndef emit(kind,**kwargs):\n    sys.stdout.write(json.dumps({\'kind\':kind,**kwargs},ensure_ascii=False,default=str)+\'\\n\')\n    sys.stdout.flush()\n\ndef get_message():\n    line=sys.stdin.readline()\n    if not line:raise EOFError(\'Notebook controller disconnected\')\n    return json.loads(line)\n\ndef parse_code(raw):\n    m=re.search(r\'```(?:python)?\\s*([\\s\\S]*?)```\',raw,re.I)\n    code=(m.group(1) if m else raw).strip()\n    if not code or len(code)>6000:return None\n    try:ast.parse(code)\n    except SyntaxError:return None\n    return code\n\ndef get_context(world):\n    desc=world.task.app_descriptions\n    apps=list(desc) if isinstance(desc,dict) else []\n    instruction=str(world.task.instruction)\n    tokens=set(re.findall(r\'[a-z]{3,}\',instruction.lower()))\n    ranked=sorted(apps,key=lambda a:(-(5*(a.lower() in instruction.lower())+len(tokens&set(re.findall(r\'[a-z]{3,}\',str(desc[a]).lower())))),a))\n    ranked=ranked[:3]\n    docs=\'\'\n    try:\n        api_doc=world.task.api_docs.function_calling()\n        if isinstance(api_doc,dict):api_doc=api_doc.get(\'tools\',api_doc.get(\'functions\',[]))\n        if isinstance(api_doc,list):\n            relevant=[]\n            for t in api_doc:\n                spec=t.get(\'function\',t) if isinstance(t,dict) else {}\n                name=str(spec.get(\'name\',\'\'))\n                if any(name.startswith(a+p) for a in ranked for p in (\'__\',\'_\',\'.\')) or name.startswith((\'supervisor\',\'api_docs\')):\n                    relevant.append(spec)\n            docs=json.dumps(relevant,default=str,ensure_ascii=False)[:14000] if relevant else \'\'\n    except Exception:\n        pass\n    if not docs:\n        try:docs=str(world.task.api_docs.compress_parameters())[:14000]\n        except Exception:docs=\'API documentation unavailable in this format; use the in-world API documentation calls.\'\n    return (\'Task instruction: \'+instruction+\'\\nApp descriptions: \'+json.dumps(desc,default=str,ensure_ascii=False)[:2200]+\'\\nRelevant API documentation: \'+docs),{\'ranked_apps\':ranked,\'doc_length\':len(docs)}\n\ndef capture_native_tool_log(task_id, experiment, folder):\n    """Only the native API-call ledger, not full AppWorld databases or ground-truth modules."""\n    source=ROOT/\'experiments\'/\'outputs\'/experiment/\'tasks\'/str(task_id)/\'logs\'/\'api_calls.jsonl\'\n    evidence={\'available\':False,\'file\':\'native_api_calls.jsonl\',\'source\':\'official AppWorld task log\',\n              \'complete\':False,\'sha256\':None,\'bytes\':None}\n    if source.is_file():\n        size=source.stat().st_size\n        evidence[\'bytes\']=size\n        if size <= 8_000_000:\n            data=source.read_bytes()\n            try:\n                records=[json.loads(line) for line in data.decode(\'utf8\').splitlines() if line.strip()]\n                if any(not isinstance(x,dict) for x in records):raise TypeError(\'API log record is not JSON object\')\n            except Exception as exc:\n                evidence[\'error\']=\'native_api_log_invalid_jsonl: \'+type(exc).__name__\n            else:\n                evidence[\'sha256\']=hashlib.sha256(data).hexdigest()\n                evidence[\'record_count\']=len(records)\n                (folder/\'native_api_calls.jsonl\').write_bytes(data)\n                evidence.update(available=True,complete=True)\n        else:\n            evidence[\'error\']=\'log_exceeds_8MB; not exported and native trace not independently inspectable\'\n    else:\n        evidence[\'error\']=\'native_api_calls_file_absent\'\n    write_json(folder/\'native_evidence_status.json\',evidence)\n    return evidence\n\ndef safe_evaluation(evaluation):\n    """Keep native pass/fail counts, not hidden tests, assertion traces or DB snapshots."""\n    passed=evaluation.get(\'passes\') or []\n    failures=evaluation.get(\'failures\') or []\n    return {\'success\':evaluation.get(\'success\'), \'difficulty\':evaluation.get(\'difficulty\'),\n            \'num_tests\':evaluation.get(\'num_tests\'), \'num_passed\':len(passed),\n            \'num_failed\':len(failures), \'source\':\'AppWorld TestTracker.to_dict\',\n            \'not_aura_verdict\':True}\n\ndef main():\n    # An app-specific environment and dependencies, never the notebook process.\n    with contextlib.redirect_stdout(sys.stderr):\n        from appworld import AppWorld, load_task_ids\n        ids=list(load_task_ids(SPLIT))\n    if not ids:raise RuntimeError(\'No benchmark IDs available for \'+SPLIT)\n    chosen=ids[:COUNT]\n    if len(chosen)!=COUNT:raise RuntimeError(\'Fewer AppWorld task IDs available than predeclared\')\n    write_json(OUT/\'appworld_task_ids.json\',{\'split\':SPLIT,\'ids\':chosen,\'selection\':\'first_n_original_dataset_order\'})\n    emit(\'ready\',task_ids=chosen)\n    rows=[]\n    for task_id in chosen:\n        folder=DEST/str(task_id);folder.mkdir(parents=True,exist_ok=True)\n        item={\'task_id\':str(task_id),\'split\':SPLIT,\'status\':\'STARTED\',\'native_evaluation_available\':False,\n              \'trajectory_type\':\'LLM_code_execution_in_native_AppWorld\'}\n        transcript=[]\n        try:\n            with contextlib.redirect_stdout(sys.stderr):\n                world=AppWorld(task_id=task_id,experiment_name=EXPERIMENT)\n            try:\n                with contextlib.redirect_stdout(sys.stderr):ctx,metadata=get_context(world)\n                item[\'agent_context_sha256\']=hashlib.sha256(ctx.encode(\'utf8\')).hexdigest()\n                item[\'context_bytes\']=len(ctx.encode(\'utf8\'))\n                messages=[{\'role\':\'system\',\'content\':SYSTEM},{\'role\':\'user\',\'content\':ctx}]\n                for step in range(STEPS):\n                    emit(\'infer\',task_id=str(task_id),step=step,messages=messages)\n                    response=get_message()\n                    if response.get(\'kind\')!=\'model_response\':raise RuntimeError(\'Missing model_response protocol\')\n                    if \'error\' in response:raise RuntimeError(\'LLM controller error: \'+str(response[\'error\'])[:300])\n                    raw=response[\'text\'];latency=response.get(\'latency_s\')\n                    code=parse_code(raw)\n                    turn={\'step\':step,\'generated\':raw,\'parsed_code\':code,\'inference_seconds\':latency}\n                    messages.append({\'role\':\'assistant\',\'content\':raw})\n                    if code is None:\n                        turn[\'result\']=\'INVALID_PYTHON_CODE\';transcript.append(turn)\n                        messages.append({\'role\':\'user\',\'content\':\'Invalid Python: provide one fenced Python block.\'})\n                        continue\n                    with contextlib.redirect_stdout(sys.stderr): result=world.execute(code)\n                    turn[\'execution_output\']=str(result)[:12000]\n                    transcript.append(turn)\n                    messages.append({\'role\':\'user\',\'content\':\'Environment output: \'+str(result)[-2000:]+\'\\nContinue task.\'})\n                    with contextlib.redirect_stdout(sys.stderr):done=bool(world.task_completed())\n                    if done:break\n                with contextlib.redirect_stdout(sys.stderr):\n                    item[\'agent_called_complete_task\']=bool(world.task_completed())\n                # Evaluate only AFTER all model decisions, never send labels into messages.\n                try:\n                    with contextlib.redirect_stdout(sys.stderr):evaluation=world.evaluate().to_dict()\n                    if not isinstance(evaluation,dict):raise TypeError(\'Native evaluator returned non-dict\')\n                    write_json(folder/\'official_appworld_evaluation_summary.json\',safe_evaluation(evaluation))\n                    item[\'official_success_field\']=evaluation.get(\'success\')\n                    item[\'native_evaluation_available\']=True\n                except Exception as exc:item[\'native_evaluation_error\']=repr(exc)\n                item[\'status\']=\'EXECUTED_EVALUATED\' if item[\'native_evaluation_available\'] else \'EXECUTED_EVALUATOR_FAILED\'\n            finally:\n                with contextlib.redirect_stdout(sys.stderr):world.close()\n        except Exception as exc:\n            item[\'status\']=\'TASK_ERROR\';item[\'error\']=repr(exc)\n            (folder/\'task_exception.txt\').write_text(traceback.format_exc(),encoding=\'utf8\')  # diagnostic only; excluded from research ZIP\n        finally:\n            write_json(folder/\'agent_code_trajectory.json\',transcript)\n            log_status=capture_native_tool_log(task_id,EXPERIMENT,folder)\n            item[\'native_api_log_available\']=log_status[\'available\']\n            item[\'native_api_log_bytes\']=log_status.get(\'bytes\')\n            write_json(folder/\'run_status.json\',item)\n            rows.append(item)\n            emit(\'task_done\',task_id=str(task_id),status=item[\'status\'],evaluated=item[\'native_evaluation_available\'],steps=len(transcript),native_log_available=item.get(\'native_api_log_available\',False))\n    with (DEST/\'summary.csv\').open(\'w\',newline=\'\',encoding=\'utf8\') as f:\n        fields=[\'task_id\',\'split\',\'status\',\'agent_called_complete_task\',\'native_evaluation_available\',\'official_success_field\',\'error\',\'native_evaluation_error\',\'trajectory_type\',\'native_api_log_available\',\'native_api_log_bytes\']\n        w=csv.DictWriter(f,fieldnames=fields,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n    control=[]\n    for task_id in chosen:\n        rec={\'task_id\':str(task_id),\'control\':\'NO_AGENT_ACTIONS\',\'native_evaluation_available\':False}\n        try:\n            with contextlib.redirect_stdout(sys.stderr):\n                with AppWorld(task_id=task_id,experiment_name=EXPERIMENT+\'_no_op_control\') as world:\n                    eval_=world.evaluate().to_dict()\n            write_json(DEST/str(task_id)/\'official_noop_control_summary.json\',safe_evaluation(eval_))\n            rec[\'native_evaluation_available\']=True\n            rec[\'official_success_field\']=eval_.get(\'success\')\n        except Exception as exc:rec[\'control_error\']=repr(exc)\n        control.append(rec)\n    write_json(DEST/\'no_op_controls.json\',control)\n    emit(\'done\',task_count=len(rows),evaluations=sum(bool(r.get(\'native_evaluation_available\')) for r in rows),controls=len(control))\n\nif __name__==\'__main__\':\n    try:main()\n    except Exception as exc:\n        (DEST/\'worker_fatal_error.txt\').write_text(traceback.format_exc(),encoding=\'utf8\')\n        emit(\'fatal\',error=repr(exc))\n        sys.exit(1)\n'

TASK_ROWS=[]
WORKER_STATUS={'attempted':False,'complete':False,'fatal':None,'events':[]}
WORKER=WORK/'appworld_worker.py'
app_out=OUT/'track_B_native_AppWorld'
app_out.mkdir(exist_ok=True)
if APPWORLD_READY and MODEL_READY:
    WORKER.write_text(WORKER_SOURCE,encoding='utf-8')
    worker_env=ENV.copy()
    worker_env.update({'AURA_RESULTS_DIR':str(OUT),'AURA_APPWORLD_SPLIT':SPLIT,
                       'AURA_APPWORLD_COUNT':str(N_TASKS),'AURA_APPWORLD_STEPS':str(MAX_STEPS),
                       'AURA_APPWORLD_EXPERIMENT':EXPERIMENT})
    WORKER_STATUS['attempted']=True
    stderr_path=OUT/'worker_stderr_private.txt'
    p=None
    try:
        with stderr_path.open('w',encoding='utf-8') as stderr:
            p=subprocess.Popen([str(APPWORLD_PY),'-u',str(WORKER)],env=worker_env,
                stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=stderr,text=True,bufsize=1)
            done=False
            for line in p.stdout:
                try:msg=json.loads(line)
                except Exception:
                    WORKER_STATUS['fatal']='Worker produced non-JSON text: '+line[:400]
                    break
                kind=msg.get('kind')
                if kind=='infer':
                    try:
                        response,seconds=llm_chat(msg['messages'])
                        answer={'kind':'model_response','text':response,'latency_s':seconds}
                    except Exception as e:
                        answer={'kind':'model_response','error':repr(e)}
                    p.stdin.write(json.dumps(answer,ensure_ascii=False)+'\n');p.stdin.flush()
                elif kind=='ready':
                    WORKER_STATUS['task_ids']=msg.get('task_ids')
                    print('Official dev task IDs',msg.get('task_ids'))
                elif kind=='task_done':
                    WORKER_STATUS['events'].append(msg)
                    print('Agent completed episode',msg.get('task_id'),msg.get('status'))
                elif kind=='fatal':
                    WORKER_STATUS['fatal']=msg.get('error')
                    print('Worker fatal:',msg.get('error'))
                elif kind=='done':done=True
            if p.poll() is None and not done:p.terminate()
            try:rc=p.wait(timeout=60)
            except subprocess.TimeoutExpired:p.kill();rc=p.wait()
            WORKER_STATUS.update(complete=(done and rc==0),exit_code=rc)
    except Exception as exc:
        WORKER_STATUS['fatal']='controller_error:'+repr(exc)
    if (app_out/'summary.csv').is_file():
        with (app_out/'summary.csv').open(newline='',encoding='utf-8') as f:
            TASK_ROWS=list(csv.DictReader(f))
    if not WORKER_STATUS['complete']:
        diag='WORKER STATUS\n'+json.dumps(WORKER_STATUS,indent=2)
        if stderr_path.exists():diag+='\n\nWORKER STDERR TAIL\n'+stderr_path.read_text(errors='replace')[-28000:]
        crash=app_out/'worker_fatal_error.txt'
        if crash.is_file():diag+='\n\nWORKER TRACEBACK\n'+crash.read_text(errors='replace')[-10000:]
        (OUT/'WORKER_DIAGNOSTIC.txt').write_text(diag[-35000:],encoding='utf-8')
else:
    print('Native agent run skipped: setup or model not ready. No official score claimed.')
(OUT/'worker_status.json').write_text(json.dumps(WORKER_STATUS,indent=2))
print('Native task records:',len(TASK_ROWS),'expected:',N_TASKS)

## 6 — Curated results ZIP, completeness audit, and exact readiness
The final archive is deliberately **small**. It includes native task IDs, generated code trajectory, official summary (not hidden tests), native API-call ledger, status, version metadata, and a readable failure traceback **only when required**. It never includes downloaded benchmark databases, official ground truth/evaluator modules, model weights, or private setup scripts.

In [ ]:
REVIEW=[]
if TASK_ROWS:
    for row in TASK_ROWS:
        tid=row.get('task_id','')
        folder=app_out/tid
        native=(folder/'official_appworld_evaluation_summary.json')
        traj=(folder/'agent_code_trajectory.json')
        status=(folder/'native_evidence_status.json')
        api=(folder/'native_api_calls.jsonl')
        findings=[]
        if row.get('status')!='EXECUTED_EVALUATED':findings.append('not_evaluated')
        if not (native.is_file() and traj.is_file() and status.is_file()):findings.append('missing_required_task_artifacts')
        if native.is_file():
            try:
                x=json.loads(native.read_text())
                if type(x.get('success')) is not bool or x.get('source')!='AppWorld TestTracker.to_dict':
                    findings.append('invalid_native_evaluation_receipt')
                if x.get('not_aura_verdict') is not True:findings.append('native_wrongly_labeled_as_aura')
            except Exception as exc:findings.append('invalid_native_json')
        if status.is_file():
            ev=json.loads(status.read_text())
            if ev.get('available') is not True:findings.append('native_api_ledger_missing')
            elif not api.is_file() or hashlib.sha256(api.read_bytes()).hexdigest()!=ev.get('sha256'):
                findings.append('native_api_ledger_hash_invalid')
        REVIEW.append({'task_id':tid,'issues':findings,'official_evaluation_available':native.is_file()})

fully_evaluated=(APPWORLD_READY and MODEL_READY and WORKER_STATUS.get('complete')
                 and len(TASK_ROWS)==N_TASKS and all(not x['issues'] for x in REVIEW))
if fully_evaluated:
    READY='READY_FOR_EXTERNAL_NATIVE_BENCHMARK_ANALYSIS'
elif FAILURE:
    READY='NOT_READY_APPWORLD_SETUP_ERROR'
elif not MODEL_READY:
    READY='NOT_READY_MODEL_ERROR'
else:
    READY='PARTIAL_NATIVE_BENCHMARK'
SUMMARY={'readiness':READY,'notebook_version':'v6-appworld-only','track_a_repeated':False,
         'appworld_ready':APPWORLD_READY,'model_ready':MODEL_READY,
         'native_task_count':len(TASK_ROWS),'native_evaluated_count':sum(row.get('status')=='EXECUTED_EVALUATED' for row in TASK_ROWS),
         'setup_failure_phase':FAILURE.get('phase') if FAILURE else None,'task_audit':REVIEW,
         'external_aura_certificates_created':False,'valid_for_held_out_score':False,
         'scientific_note':'Authentic AppWorld official dev scores; independent AURA AppWorld verification adapter not built.'}
(OUT/'EXPERIMENT_SUMMARY.json').write_text(json.dumps(SUMMARY,indent=2))

export_root=WORK/'curated_export';export_root.mkdir()
allowed_root=['config.json','setup_checklist.json','setup_status.json','native_evaluator_preflight.json',
              'model_status.json','worker_status.json','EXPERIMENT_SUMMARY.json']
allowed_native=['run_status.json','agent_code_trajectory.json','official_appworld_evaluation_summary.json',
                'native_evidence_status.json','native_api_calls.jsonl','official_noop_control_summary.json']
missing=[]
def add(path,dst,required=False):
    path=Path(path)
    if not path.is_file():
        if required:missing.append(dst)
        return
    target=export_root/dst
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(path,target)
for k in allowed_root:add(OUT/k,k,required=True)
if FAILURE:add(OUT/'SETUP_DIAGNOSTIC.txt','SETUP_DIAGNOSTIC.txt',required=True)
if not MODEL_READY and APPWORLD_READY:add(OUT/'MODEL_DIAGNOSTIC.txt','MODEL_DIAGNOSTIC.txt')
if not WORKER_STATUS.get('complete') and WORKER_STATUS.get('attempted'):
    add(OUT/'WORKER_DIAGNOSTIC.txt','WORKER_DIAGNOSTIC.txt',required=True)
for k in ['summary.csv','no_op_controls.json']:
    add(app_out/k,'track_B/'+k,required=WORKER_STATUS.get('complete'))
add(OUT/'appworld_task_ids.json','track_B/appworld_task_ids.json',required=WORKER_STATUS.get('complete'))
for row in TASK_ROWS:
    tid=row['task_id']
    for name in allowed_native:
        add(app_out/tid/name,'track_B/'+tid+'/'+name,
            required=bool(row.get('status')=='EXECUTED_EVALUATED' and name!='official_noop_control_summary.json'))

if missing:
    SUMMARY['readiness']='NOT_READY_MISSING_EXPORT_ARTIFACTS'
    SUMMARY['missing_exports']=missing
    (export_root/'EXPERIMENT_SUMMARY.json').write_text(json.dumps(SUMMARY,indent=2))
    READY=SUMMARY['readiness']

def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest=[{'path':p.relative_to(export_root).as_posix(),'bytes':p.stat().st_size,'sha256':sha(p)}
          for p in sorted(export_root.rglob('*')) if p.is_file()]
(export_root/'FILE_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
(export_root/'README.txt').write_text(
    'AURA Stage 8.1 v6 recovery: AppWorld-native dev agent pilot ONLY.\n'
    'Earlier local AURA runs are intentionally not included or rerun.\n'
    'Official AppWorld results are not independent AURA certificates.\n'
    'If setup fails, consult SETUP_DIAGNOSTIC.txt.\n'
    'No model weights, benchmark databases, hidden evaluators or installation logs.\n')

with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for p in sorted(export_root.rglob('*')):
        if p.is_file():archive.write(p,p.relative_to(export_root).as_posix())
with zipfile.ZipFile(ZIP_PATH,'r') as archive:
    assert archive.testzip() is None
    names=set(archive.namelist())
    assert len(names)==len(archive.namelist())
    proof=json.loads(archive.read('FILE_MANIFEST.json'))
    assert names=={e['path'] for e in proof}|{'FILE_MANIFEST.json','README.txt'}
    for m in proof:
        raw=archive.read(m['path'])
        assert len(raw)==m['bytes'] and hashlib.sha256(raw).hexdigest()==m['sha256']
    forbidden=('ground_truth','/dbs/','appworld_worker.py','installation.txt','venv/',
               'site-packages/','worker_stderr_private.txt','full_evaluator')
    assert not any(any(t in name for t in forbidden) for name in names)
print('STAGE 8.1 APPWORLD READINESS:',READY)
print('Actual AppWorld episodes:',len(TASK_ROWS),'of',N_TASKS)
print('Results ZIP:',ZIP_PATH,'files:',len(names),'bytes:',ZIP_PATH.stat().st_size)
print('Download ZIP regardless of success; it contains failure traceback if setup stops.')